# UAV 路径规划 — 交互式操作面板

**使用规则：每次打开 Notebook 后，必须先运行第一格（初始化），之后所有格均可独立点击运行。**

---

In [ ]:
# ★ 初始化（每次打开 Notebook 后必须首先运行此格）
from pipeline_api import *

---
## 一、运行控制

| 指令 | 说明 |
|---|---|
| `run()` | 完整运行全部 5 个阶段 |
| `run("3-5")` | 从阶段3开始运行到阶段5 |
| `run(3)` | 只运行阶段3 |
| `run("2", force=True)` | 强制重新计算阶段2（忽略已有检查点） |
| `stage1()` ~ `stage5()` | 单独运行某一阶段 |
| `stage3(force=True, quality_threshold=0.7)` | 带参数运行阶段3 |

In [ ]:
run()   # 完整运行（阶段1 → 5）

In [ ]:
run("3-5")   # 从阶段3恢复运行（需要阶段2的检查点）

In [ ]:
stage1()   # 单独运行阶段1

In [ ]:
stage2()   # 单独运行阶段2（耗时最长，建议完成后不要轻易重跑）

In [ ]:
stage3()   # 单独运行阶段3（使用默认覆盖阈值 0.85）

In [ ]:
stage3(force=True, quality_threshold=0.7)   # 强制重跑，调低阈值（航点更少）

In [ ]:
stage4()   # 单独运行阶段4（多机任务分配 + TSP）

In [ ]:
stage5()   # 单独运行阶段5（A* 避障 + 轨迹 CSV 导出）

---
## 二、状态与结果查看

| 指令 | 说明 |
|---|---|
| `status()` | 查看所有检查点文件和输出文件的存在状态 |
| `summary()` | 查看各阶段的关键结果数字（航点数、路径长度等）|
| `inspect_csv(1)` | 查看 UAV 1 的轨迹 CSV 统计信息 |
| `inspect_csv(2, n_rows=10)` | 查看 UAV 2 的 CSV，预览前10行 |

In [ ]:
status()   # 查看所有检查点和输出文件状态

In [ ]:
summary()   # 查看各阶段结果摘要（航点数、路径长度、CSV 统计等）

In [ ]:
inspect_csv(1)   # 查看 UAV 1 的轨迹 CSV 统计

In [ ]:
inspect_csv(2, n_rows=10)   # 查看 UAV 2 的 CSV，预览前10行

---
## 三、参数调整

| 指令 | 说明 |
|---|---|
| `show_config()` | 显示全部配置参数（分类展示）|
| `set_config(KEY=val, ...)` | 运行时修改任意参数（立即生效）|
| `reset_config()` | 恢复 config.py 中的默认值 |

> **调参后重跑范围参考：**  
> 修改成像/安全参数 → `reset(2)` 后 `run("2-5")`  
> 修改多机参数 → `reset(4)` 后 `run("4-5")`  
> 修改轨迹参数 → 直接 `stage5()`

In [ ]:
show_config()   # 显示全部配置参数

In [ ]:
set_config(FOV_DEG=80, CAMERA_DISTANCE=4.0)   # 修改成像参数（需重跑阶段2）

In [ ]:
set_config(FLIGHT_SAFE_RADIUS=1.5, HOVER_TIME=8.0)   # 修改安全距离和悬停时长

In [ ]:
set_config(NUM_UAVS=2, TAKEOFF_POINTS=[[20,0,0.5],[-20,0,0.5]])   # 改为双机

In [ ]:
reset_config()   # 恢复所有参数为 config.py 默认值

---
## 四、检查点管理

| 指令 | 说明 |
|---|---|
| `reset()` | 清除全部检查点和内存缓存 |
| `reset(3)` | 仅清除阶段3及之后的检查点（阶段1、2保留）|
| `save_snapshot("name")` | 将当前检查点打包为命名快照 |
| `load_snapshot("name")` | 从命名快照恢复检查点 |
| `list_snapshots()` | 列出所有已保存快照 |

In [ ]:
reset()   # 清除全部检查点（从头重跑时使用）

In [ ]:
reset(3)   # 仅清除阶段3及之后（调整阶段3参数后使用）

In [ ]:
save_snapshot("fov90_radius3")   # 保存当前实验结果为快照

In [ ]:
load_snapshot("fov90_radius3")   # 从快照恢复（可继续运行下游阶段）

In [ ]:
list_snapshots()   # 查看所有已保存快照

---
## 五、典型调试流程示例

以下是几种常用场景，按顺序运行对应格即可。

In [ ]:
# 场景A：首次运行完整流程
run()

In [ ]:
# 场景B：调整覆盖阈值，只重跑阶段3及之后（阶段1、2无需重跑）
reset(3); run("3-5")

In [ ]:
# 场景C：修改相机视场角，重跑阶段2及之后
set_config(FOV_DEG=80); reset(2); run("2-5")

In [ ]:
# 场景D：保存当前结果，切换参数对比实验
save_snapshot("baseline"); set_config(FOV_DEG=120); reset(2); run("2-5")

In [ ]:
# 场景E：恢复基线实验结果，继续调试阶段5
load_snapshot("baseline"); stage5()